In [ ]:
# Section 1: Import Libraries
# Import Python libraries for file handling, manifest parsing, HTTP requests, and website API interaction.
import os
from pathlib import Path
import json

try:
    import openpyxl
except ImportError:
    raise ImportError('openpyxl is required to read the image sourcing manifest XLSX file.')

# Section 2: Load Image Sourcing Manifest
# Load the manifest file from disk or repository, reading JSON/CSV/YAML content to memory.
manifest_path = Path('SkinVault_Image_Sourcing_Manifest.xlsx')
if not manifest_path.exists():
    raise FileNotFoundError(f'Manifest file not found: {manifest_path}')

workbook = openpyxl.load_workbook(manifest_path)
print('Loaded workbook sheets:', workbook.sheetnames)

# Section 3: Inspect Manifest Structure
# Explore manifest keys, image source fields, and product identifiers to understand how images are linked to products.
worksheet = workbook.active
header_row = [cell.value for cell in next(worksheet.iter_rows(min_row=1, max_row=1, values_only=True))]
print('Header row:', header_row)
rows = list(worksheet.iter_rows(min_row=2, values_only=True))
print('Sample rows:')
for row in rows[:10]:
    print(row)

# Section 4: Build Product-to-Image Mapping
# Create a mapping between exact product IDs or SKUs and their associated image URLs or file paths.
product_image_map = {}

# Attempt to detect common manifest columns.
column_map = {name.lower().strip(): idx for idx, name in enumerate(header_row) if name}
print('Detected columns:', column_map)

# Potential keys in the manifest for product name, SKU, image path, image URL.
product_keys = ['product', 'product name', 'sku', 'item name', 'item']
image_keys = ['image', 'image url', 'image path', 'image location', 'photo', 'img']

product_col = next((column_map[k] for k in product_keys if k in column_map), None)
image_col = next((column_map[k] for k in image_keys if k in column_map), None)
print('Using product_col=', product_col, 'image_col=', image_col)

if product_col is None or image_col is None:
    raise ValueError('Could not detect product or image columns in the manifest.')

for row in rows:
    if not row:
        continue
    product_value = row[product_col]
    image_value = row[image_col]
    if product_value and image_value:
        product_name = str(product_value).strip()
        image_src = str(image_value).strip()
        product_image_map.setdefault(product_name, []).append(image_src)

print('Built mapping for', len(product_image_map), 'products')
for product, images in list(product_image_map.items())[:10]:
    print(product, images)

# Section 5: Download or Collect Image Files
# Download referenced images or resolve local image paths, saving them to a working directory for upload.
# Note: This notebook collects image sources; actual download behavior depends on whether URLs or local paths are present.
images_dir = Path('assets/products')
images_dir.mkdir(parents=True, exist_ok=True)

downloaded = []
for product, images in product_image_map.items():
    for image_src in images:
        if image_src.startswith('http://') or image_src.startswith('https://'):
            print('Remote image source detected:', image_src)
        else:
            image_path = Path(image_src)
            if image_path.exists():
                destination = images_dir / image_path.name
                if not destination.exists():
                    destination.write_bytes(image_path.read_bytes())
                downloaded.append(destination)
            else:
                print('Local image path not found:', image_src)

print('Collected images:', len(downloaded))

# Section 6: Attach Images to Website Products
# Use the website's product API or database interface to add images to the exact product records.
# This notebook will prepare a plan for updating product HTML pages.
website_files = ['about.html', 'beauty-from-within.html', 'bundles.html', 'concerns.html', 'contact.html', 'journal.html', 'shop.html', 'skinvault-by-chi.html']
print('Candidate website files:', website_files)

# Section 7: Verify Product Image Updates
# Query the website product data to confirm each product has the expected images attached.
for page in website_files:
    page_path = Path(page)
    if page_path.exists():
        content = page_path.read_text(encoding='utf-8')
        found = any(product in content for product in product_image_map.keys())
        if found:
            print(f'Found at least one product reference in {page}')
        else:
            print(f'No product references found in {page}')
    else:
        print('Missing page file:', page)
